<a href="https://colab.research.google.com/github/zeets13/Flaggr_Project/blob/main/Notebook/3_Binary_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Preparation**

Datset Loading

In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset
dataset = load_dataset(
    "ucberkeley-dlab/measuring-hate-speech",
    "default"
)
df = dataset["train"].to_pandas()
print("Rows:", len(df))
print("Columns:", len(df.columns))

README.md:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/135556 [00:00<?, ? examples/s]

Rows: 135556
Columns: 143


Data Exploring

In [ ]:
print("Original rows:", len(df))

print(df.columns.tolist())

Original rows: 135556
['comment_id', 'annotator_id', 'platform', 'sentiment', 'respect', 'insult', 'humiliate', 'status', 'dehumanize', 'violence', 'genocide', 'attack_defend', 'hatespeech', 'hate_speech_score', 'text', 'infitms', 'outfitms', 'annotator_severity', 'std_err', 'annotator_infitms', 'annotator_outfitms', 'hypothesis', 'target_race_asian', 'target_race_black', 'target_race_latinx', 'target_race_middle_eastern', 'target_race_native_american', 'target_race_pacific_islander', 'target_race_white', 'target_race_other', 'target_race', 'target_religion_atheist', 'target_religion_buddhist', 'target_religion_christian', 'target_religion_hindu', 'target_religion_jewish', 'target_religion_mormon', 'target_religion_muslim', 'target_religion_other', 'target_religion', 'target_origin_immigrant', 'target_origin_migrant_worker', 'target_origin_specific_country', 'target_origin_undocumented', 'target_origin_other', 'target_origin', 'target_gender_men', 'target_gender_non_binary', 'target_ge

In [ ]:
print(df[[
    "comment_id",
    "text",
    "hatespeech",
    "hate_speech_score"
]].head())

   comment_id                                               text  hatespeech  \
0       47777  Yes indeed. She sort of reminds me of the elde...         0.0   
1       39773  The trans women reading this tweet right now i...         0.0   
2       47101  Question: These 4 broads who criticize America...         2.0   
3       43625  It is about time for all illegals to go back t...         0.0   
4       12538  For starters bend over the one in pink and kic...         2.0   

   hate_speech_score  
0              -3.90  
1              -6.52  
2               0.36  
3               0.26  
4               1.54  


HateSpeech Labels

In [ ]:
print(df["hatespeech"].value_counts().sort_index())

hatespeech
0.0    80624
1.0     8911
2.0    46021
Name: count, dtype: int64


In [ ]:
print(df["hatespeech"].value_counts().sort_index())

print(
    df.groupby("comment_id")["hatespeech"]
      .apply(list)
      .head(20)
)

hatespeech
0.0    80624
1.0     8911
2.0    46021
Name: count, dtype: int64
comment_id
1     [0.0, 0.0, 0.0, 0.0]
2          [2.0, 2.0, 0.0]
3          [1.0, 2.0, 2.0]
4               [1.0, 2.0]
5          [0.0, 0.0, 0.0]
6               [0.0, 0.0]
7          [0.0, 2.0, 0.0]
8                    [0.0]
10         [0.0, 2.0, 2.0]
11              [2.0, 1.0]
12              [0.0, 2.0]
13              [0.0, 0.0]
15                   [1.0]
17                   [0.0]
18         [0.0, 0.0, 0.0]
19              [0.0, 0.0]
22              [2.0, 0.0]
23         [2.0, 2.0, 2.0]
24                   [0.0]
25                   [0.0]
Name: hatespeech, dtype: object


Aggregating multiple annotators scoring using majority count

In [ ]:
from collections import Counter

def majority_label(values):
    counts = Counter(values)

    max_count = max(counts.values())

    winners = [
        label for label, count in counts.items()
        if count == max_count
    ]

    if len(winners) != 1:
        return None

    return winners[0]

Aggregation

In [ ]:
df_binary = (
    df.groupby("comment_id")
      .agg({
          "text": "first",
          "hatespeech": list
      })
      .reset_index()
)

df_binary["hate_speech_label"] = (
    df_binary["hatespeech"]
    .apply(majority_label)
)

In [ ]:
print(
    df_binary["hate_speech_label"]
    .value_counts(dropna=False)
    .sort_index()
)

hate_speech_label
0.0    24755
1.0     1291
2.0     8328
NaN     5191
Name: count, dtype: int64


Converting to binary dataset

In [ ]:
df_binary = df_binary[
    df_binary["hate_speech_label"].isin([0, 2])
].copy()

Removed the samples that didnt have a clear label

In [ ]:
df_binary["binary_label"] = (
    df_binary["hate_speech_label"] == 2
).astype(int)

In [ ]:
print(
    df_binary["binary_label"]
    .value_counts()
    .sort_index()
)

binary_label
0    24755
1     8328
Name: count, dtype: int64


Taking 15k samples from 39k+ data using stratified splitting strategy

In [ ]:
from sklearn.model_selection import train_test_split

df_sample, _ = train_test_split(
    df_binary,
    train_size=15000,
    stratify=df_binary["binary_label"],
    random_state=42
)

df_sample = df_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Dataset size:", len(df_sample))

print(
    df_sample["binary_label"]
    .value_counts()
    .sort_index()
)

print(
    df_sample["binary_label"]
    .value_counts(normalize=True)
)

Dataset size: 15000
binary_label
0    11224
1     3776
Name: count, dtype: int64
binary_label
0    0.748267
1    0.251733
Name: proportion, dtype: float64


Splitting into train/valid/test (80:10:10)

In [ ]:
train_df, temp_df = train_test_split(
    df_sample,
    test_size=0.20,
    stratify=df_sample["binary_label"],
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["binary_label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))

Train: 12000
Validation: 1500
Test: 1500


Checking out the split size

In [ ]:
print("\n===== TRAIN =====")
print(train_df["binary_label"].value_counts())
print(train_df["binary_label"].value_counts(normalize=True))

print("\n===== VALIDATION =====")
print(valid_df["binary_label"].value_counts())
print(valid_df["binary_label"].value_counts(normalize=True))

print("\n===== TEST =====")
print(test_df["binary_label"].value_counts())
print(test_df["binary_label"].value_counts(normalize=True))


===== TRAIN =====
binary_label
0    8979
1    3021
Name: count, dtype: int64
binary_label
0    0.74825
1    0.25175
Name: proportion, dtype: float64

===== VALIDATION =====
binary_label
0    1122
1     378
Name: count, dtype: int64
binary_label
0    0.748
1    0.252
Name: proportion, dtype: float64

===== TEST =====
binary_label
0    1123
1     377
Name: count, dtype: int64
binary_label
0    0.748667
1    0.251333
Name: proportion, dtype: float64


Saving the splitted dataset

In [ ]:
train_df.to_csv(
    "/content/drive/MyDrive/hate_speech_data/binary_label_train_15000.csv",
    index=False
)

valid_df.to_csv(
    "/content/drive/MyDrive/hate_speech_data/binary_label_validation_15000.csv",
    index=False
)

test_df.to_csv(
    "/content/drive/MyDrive/hate_speech_data/binary_label_test_15000.csv",
    index=False
)

**Baseline Model for binary classification (TF-IDF with Logistic Regression)**

Text preprocessing before training baseline model

In [ ]:
import re

def preprocess_text(text):
    text = str(text)

    text = re.sub(r"<[^>]+>", " ", text)

    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

In [ ]:
train_texts = train_df["text"].apply(preprocess_text)
valid_texts = valid_df["text"].apply(preprocess_text)
test_texts = test_df["text"].apply(preprocess_text)

y_train = train_df["binary_label"]
y_valid = valid_df["binary_label"]
y_test = test_df["binary_label"]

Using TF-IDF as baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=50000
)

In [ ]:
X_train = tfidf.fit_transform(train_texts)

X_valid = tfidf.transform(valid_texts)

X_test = tfidf.transform(test_texts)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

Evaluating on validation set

In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score
)

valid_predictions = lr_model.predict(X_valid)

valid_probabilities = lr_model.predict_proba(X_valid)[:, 1]

print("===== VALIDATION RESULTS =====")

print(
    classification_report(
        y_valid,
        valid_predictions,
        target_names=["Non-Hate", "Hate"],
        digits=4
    )
)

print(
    "Validation Accuracy:",
    accuracy_score(y_valid, valid_predictions)
)

print(
    "Validation ROC-AUC:",
    roc_auc_score(y_valid, valid_probabilities)
)

===== VALIDATION RESULTS =====
              precision    recall  f1-score   support

    Non-Hate     0.8949    0.8806    0.8877      1122
        Hate     0.6616    0.6931    0.6770       378

    accuracy                         0.8333      1500
   macro avg     0.7783    0.7868    0.7823      1500
weighted avg     0.8361    0.8333    0.8346      1500

Validation Accuracy: 0.8333333333333334
Validation ROC-AUC: 0.8813932980599647


Evaluating baseline model on test set

In [ ]:
test_predictions = lr_model.predict(X_test)

test_probabilities = lr_model.predict_proba(X_test)[:, 1]

print("===== TEST RESULTS =====")

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["Non-Hate", "Hate"],
        digits=4
    )
)

print(
    "Test Accuracy:",
    accuracy_score(y_test, test_predictions)
)

print(
    "Test ROC-AUC:",
    roc_auc_score(y_test, test_probabilities)
)

===== TEST RESULTS =====
              precision    recall  f1-score   support

    Non-Hate     0.8935    0.8816    0.8875      1123
        Hate     0.6607    0.6870    0.6736       377

    accuracy                         0.8327      1500
   macro avg     0.7771    0.7843    0.7805      1500
weighted avg     0.8350    0.8327    0.8337      1500

Test Accuracy: 0.8326666666666667
Test ROC-AUC: 0.8739214542328124


Example of predictions

In [ ]:
def predict_text(text):

    processed = preprocess_text(text)

    X = tfidf.transform([processed])

    prediction = lr_model.predict(X)[0]
    probability = lr_model.predict_proba(X)[0][1]

    label = "Hate" if prediction == 1 else "Non-Hate"

    print("Text:", text)
    print("Prediction:", label)
    print(f"Hate probability: {probability:.4f}")

In [ ]:
predict_text("Hello, how are you today?")

Text: Hello, how are you today?
Prediction: Non-Hate
Hate probability: 0.4152


In [ ]:
predict_text("Hello you dumbo")

Text: Hello you dumbo
Prediction: Non-Hate
Hate probability: 0.4815


In [ ]:
predict_text("dumb bitch")

Text: dumb bitch
Prediction: Hate
Hate probability: 0.9283


In [ ]:
predict_text("nigga just go kill yourself")

Text: nigga just go kill yourself
Prediction: Hate
Hate probability: 0.9433


In [ ]:
predict_text("shut your stupid mouth")

Text: shut your stupid mouth
Prediction: Hate
Hate probability: 0.7074
